In [ ]:
%pip install scipy

In [ ]:
import json
import numpy as np
import os
import pandas as pd
from scipy.stats import zscore

In [ ]:
# import os
# os.environ["TQDM_DISABLE"] = "0"

# Add honesty judgements (if not there yet)

In [ ]:
# llm configs, keyvault connection + fixing the HF cache
from llm_eval.utils.setup_utils import benchmark_data_folder, get_gpt_secrets, get_hf_secrets
from llm_eval.language_models import LLMRouter

gpt_secrets = get_gpt_secrets()

gpt_judge_params = {
    "temperature": 0,
    "top_p": 1,
    "frequency_penalty": 0,
    "presence_penalty": 0,
    "n": 1,
    "max_tokens": 20,
}

# Separate judge model to limig tokens and avoid accumulating codecarbon duration
gpt_4o_mini = LLMRouter.get_model(
    provider="azure",
    model_name="gpt-4o-mini",
    api_endpoint=gpt_secrets["API_ENDPOINT"],
    api_key=gpt_secrets["API_KEY"],
    api_version=gpt_secrets["API_VERSION"],
    params=gpt_judge_params,
    uses_api=True,
)

hf_secrets = get_hf_secrets()
hf_judge_infenerence_params = {
    "do_sample": False,
    "temperature": 0,
    "top_k": 0,
    "top_p": 1.0,
    "repetition_penalty": 1.0,
    "num_return_sequences": 1,
    "max_new_tokens": 20,
}

hf_judge_object_params = {
    # "provider": "vllm",
    "provider": "huggingface",
    "hf_token": hf_secrets["HF_TOKEN"],
    "params": hf_judge_infenerence_params,
    "uses_api": False,
}

qwen_small = LLMRouter.get_model(
    model_name="qwen-8b",
    **hf_judge_object_params,
)

gemma_small = LLMRouter.get_model(
    model_name="gemma-12b-instruct",
    **hf_judge_object_params,
)

mistral = LLMRouter.get_model(
    model_name="mistral-7b-instruct-v0.3",
    **hf_judge_object_params,
)

tinyllama = LLMRouter.get_model(
    model_name="tiny-llama",
    **hf_judge_object_params,
)

In [ ]:
from llm_eval.benchmarks.honesty.honest_city_eval import HonestCityEvaluator

evaluator = HonestCityEvaluator(judge_llms=[gpt_4o_mini, qwen_small, gemma_small])

In [ ]:
# import tqdm.notebook
# import sys
# sys.modules['tqdm'] = tqdm.notebook  # Redirect tqdm imports to notebook tqdm

In [ ]:
# %pip install ninja

In [ ]:
import sys
!{sys.executable} -m pip install ninja

In [ ]:
import os
import subprocess

# Get the conda environment's bin path
conda_bin = subprocess.check_output(['conda', 'info', '--base'], text=True).strip() + '/envs/llm_eval/bin'

# Add to PATH if not already there
if conda_bin not in os.environ['PATH']:
    os.environ['PATH'] = conda_bin + ':' + os.environ['PATH']

# Verify ninja is now accessible
print("Ninja path:", subprocess.check_output(['which', 'ninja'], text=True).strip())

In [ ]:
import sys
print("Python executable:", sys.executable)
print("Python path:", sys.path[0])

import subprocess
try:
    ninja_path = subprocess.check_output(['which', 'ninja'], text=True).strip()
    print("Ninja found at:", ninja_path)
except:
    print("Ninja not found in PATH")
    print("Current PATH:", os.environ.get('PATH', ''))

In [ ]:
!export VLLM_USE_TRITON_FLASH_ATTN=0

In [ ]:
# runs = [
#     'eurollm_large', 'eurollm_small',
#     'falcon_small',
#     # 'gemma_large',
#     'gemma_large_gossa', 'gemma_small', 'gemma_small_summary', 'gemma_small_tiny',
#     'gpt4o', 'gpt4o_mini',
#     'llama_small',
#     'mistral_tiny', 'mistral_small', 'mistral_small_summary',
#     'olmo_large', 'olmo_small', 'olmo_large_summary',
#     'phi_mini',
#     'qwen_large', 'qwen_small', 'qwen_small_nothinking', 'qwen_large_nothinking',
#     'tinyllama',
# ]
# results_files = ["leaderboard_" + run for run in runs]

# results_files = ["../leaderboard"]
# results_files = ["../leaderboard_honesty_mistral_llama_small"]
results_files = ["../leaderboard_honesty_all_200"]
# results_files = ["../results_leaderboard_final_h100_2025_06/leaderboard_honesty_test"]


for results_file in results_files:
    results = json.loads(open(results_file, "rb").read())

    for entry in results:
        metadata = entry['metadata']
        benchmark = metadata["benchmark"]["name"]
        print(benchmark)

        if benchmark == "HonestCity":
            print(f"Processing {metadata['llm']['model_name']}")
            responses = entry["benchmark_results"]["run_output"]
            metrics = evaluator.evaluate(responses)
            entry["benchmark_results"]["score"] = metrics

        # Dump after every entry to avoid losing judgements
        with open(results_file, "w") as f:
            json.dump(results, f, indent=4, default=str)

# Read up results to generate the leaderboard scores

#### Option 1: Single file

In [ ]:
results_file = "../leaderboard"
results = json.loads(open(results_file, "rb").read())
# results

#### Option 2: Combine many files

In [ ]:
# shared_folder = "/home/azureuser/cloudfiles/code/shared_data/results/"
shared_folder = "../results_leaderboard_final_h100_2025_06"
# os.listdir(shared_folder)

In [ ]:
runs = [
    'eurollm_large', 'eurollm_small',
    'falcon_small',
    # 'gemma_large',
    'gemma_large_gossa', 'gemma_small', 'gemma_small_summary', 'gemma_small_tiny',
    'gpt4o', 'gpt4o_mini',
    'llama_small',
    'mistral_tiny', 'mistral_small', 'mistral_small_summary',
    'olmo_large', 'olmo_small', 'olmo_large_summary',
    'phi_mini',
    'qwen_large', 'qwen_small', 'qwen_small_nothinking', 'qwen_large_nothinking',
    'tinyllama',
]

# results_files = ["leaderboard_" + run for run in runs] + ["../leaderboard_honesty_all_200"]
results_files = ["../leaderboard_rerun_benches", "../leaderboard_honesty_all_200"]

In [ ]:
results = sum([json.loads(open(f"{shared_folder}/{file}", "rb").read()) for file in results_files], [])

# Add costs

In [ ]:
from collections import defaultdict
import tiktoken
from datetime import datetime
import sys

# Included benchmarks
costs_included_benchmarks = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
    "CNNDailyMail",
    "XSum",
]

# API pricing table (€ per 1k tokens as of 26 August 2025)
# considering an exchange rate of 1 USD = 0.8679 EUR
api_model_pricing = {
    "gpt-4o": {"input": 0.0026253, "output": 0.0105012},
    "gpt-4o-mini": {"input": 0.00014320, "output": 0.0005728},
}

# Current GPU rates on Azure (in € per hour as of 26 August 2025, 
# considering an exchange rate of 1 USD = 0.8679 EUR
gpu_hourly_rates = {
    "Tesla T4": 0.5728,
    "NVIDIA H100 NVL": 7.8805,
    "Tesla V100-PCIE-16GB": 3.3154,
}

def parse_duration(start, end):
    """Parse duration in seconds per benchmark run."""
    fmt = "%Y-%m-%dT%H:%M:%SZ"
    start_time = datetime.strptime(start, fmt)
    end_time = datetime.strptime(end, fmt)
    return (end_time - start_time).total_seconds()

def count_tokens(model_name, text):
    """Count tokens using tiktoken for a given model and text."""
    try:
        enc = tiktoken.encoding_for_model(model_name)
        return len(enc.encode(text))
    except Exception:
        return 0

results_by_model = defaultdict(lambda: {"total_cost": 0.0, "num_entries": 0})

def get_costs(entry):
    # Process entries in leaderboard dataframe
    try:
        metadata = entry["metadata"]
        model = metadata["llm"]["model_name"]
        benchmark = metadata["benchmark"]["name"]
        if benchmark not in costs_included_benchmarks: return -1

        n_tokens = metadata.get("n_tokens")
        run_output = entry.get("benchmark_results", {}).get("run_output", [])
        n_samples = metadata.get("n_samples", 1)
        run = metadata.get("run")
        device = run.get("system", {}).get("device_info", {}).get("gpu", {}).get("device_name")
        start_time, end_time = run.get("timestamp_bench_start"), run.get("timestamp_bench_end")

        if model in api_model_pricing:
            pricing = api_model_pricing[model]
            n_input = n_tokens.get("n_input_tokens") if isinstance(n_tokens, dict) else None
            n_output = n_tokens.get("n_output_tokens") if isinstance(n_tokens, dict) else None

            if n_input is None or n_output is None:
                inputs = [metadata["benchmark"]["prompt_template"] + (r.get("prompt") or r.get("source") or "") for r in run_output]
                outputs = [r.get("response") for r in run_output]
                n_input = sum(count_tokens(model, p) for p in inputs if isinstance(p, str))
                n_output = sum(count_tokens(model, o) for o in outputs if isinstance(o, str))

            cost = n_input * pricing["input"] / 1000 + n_output * pricing["output"] / 1000

        else:
            # duration = parse_duration(start_time, end_time)
            duration = metadata["code_carbon"]["duration"]
            gpu_rate = gpu_hourly_rates.get(device)
            if duration and gpu_rate:
                cost = duration * gpu_rate / 3600
            else:
                return -1

        # Price per prompt
        return cost / n_samples
    except Exception as e:
        print(f"Skipping due to error: {e}")
        return -1

In [ ]:
def get_score(entry):
    if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]:
        return entry["benchmark_results"]["score"]["acc"]
    elif entry["metadata"]["benchmark"]["name"] in ["TinyMMLU", "TinyARC", "TinyTruthfulQA"]:
        # return entry["benchmark_results"]["score"]["acc"]
        # return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["irt"],
        # return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["pirt"],
        return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["gpirt"]
        # return gpirt if not np.isnan(gpirt) else -1
    elif entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]:
        return entry["benchmark_results"]["score"]["sari"]["sari"]
    elif entry["metadata"]["benchmark"]["name"] in ["CNNDailyMail", "XSum"]:
        bert_score = entry["benchmark_results"]["score"]["bert_score"]
        return np.mean(bert_score["f1"]) if bert_score else 0
    elif entry["metadata"]["benchmark"]["name"] == "HonestCity":
        try:
            return entry["benchmark_results"]["score"].get("metrics", {}).get("mean_honesty_rate", -1) if entry["benchmark_results"]["score"] else -1
        except Exception as e:
            print(e)
            print(entry["benchmark_results"].keys())
    else:
        return -1

filtered_data = [
    {
        "runtime": entry["metadata"]["run"]["time_bench_total"],
        "llm_name": entry["metadata"]["llm"]["model_name"],
        "bench_name": entry["metadata"]["benchmark"]["name"],
        "environment_info_co2": entry["metadata"]["code_carbon"]["emissions"] if entry["metadata"]["code_carbon"] else -1,
        "duration": entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
        "environment_info_energy": entry["metadata"]["code_carbon"]["energy_consumed"] if entry["metadata"]["code_carbon"] else -1,
        "score": get_score(entry),
        "costs": get_costs(entry),
        "energy_per_time": entry["metadata"]["code_carbon"]["energy_consumed"] / entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
    } 
    # for entry in results
    for entry in results if "ARCHIVE" not in entry["metadata"]["llm"]["model_name"]
]

In [ ]:
df = pd.DataFrame(filtered_data)

# Pivot the DataFrame to create a multi-level column index
pivot_df = df.pivot_table(
    index='llm_name',
    columns='bench_name',
    values=['score', 'environment_info_co2', 'runtime', 'duration', 'environment_info_energy', 'costs', 'energy_per_time'],
    aggfunc='first'
)

# Reorder the columns to have a multi-level index
pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)

In [ ]:
pivot_df[['costs', 'energy_per_time', 'environment_info_energy', 'duration']]

In [ ]:
# order = ["ARC-NL", "MMLU-NL", "INT_Duidelijke_Taal-detailed", "INT_Duidelijke_Taal-simple", "AmsterdamSimplification-detailed", "AmsterdamSimplification-simple", "CNNDailyMail", "XSum"]
# order = ["MMLU-NL", "ARC-NL", "TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]
order = ["TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum", "HonestCity"]

In [ ]:
pivot_df["score"].reindex(columns=order)

### Inspect data

In [ ]:
pivot_df["environment_info_co2"].reindex(columns=order) * 1000

In [ ]:
pivot_df["environment_info_energy"].reindex(columns=order) * 1000

In [ ]:
pivot_df["duration"].reindex(columns=order)

In [ ]:
pivot_df["score"].reindex(columns=order)

In [ ]:
pivot_df["duration"]

In [ ]:
pivot_df["duration"].reindex(columns=order)

In [ ]:
# pivot_df["environment_info_co2"] = pivot_df["environment_info_co2"].reindex(columns=order)
pivot_df["score"] = pivot_df["score"].reindex(columns=order)
pivot_df["duration"] = pivot_df["duration"].reindex(columns=order)

# Map to categories
-------------------

# Environmental Impact

In [ ]:
def get_env_cat_emissions(score):
    if score < 0:
        return score
    if score < 0.0010:
        return 5
    elif score < 0.0013:
        return 4
    elif score < 0.0016:
        return 3
    elif score < 0.0020:
        return 2
    else:
        return 1

pivot_df.loc[:, ("environment_info_co2", "co2_emissions_mean")] = pivot_df["environment_info_co2"].mean(axis=1)
pivot_df.loc[:, ("environment_info_co2", "co2_emissions_category")] = list(map(lambda x: get_env_cat_emissions(x), pivot_df["environment_info_co2"]["co2_emissions_mean"]))

In [ ]:
# def get_env_cat_energy(score):
#     if score < 0:
#         return score
#     if score < 0.015:
#         return 5
#     elif score < 0.025:
#         return 4
#     elif score < 0.05:
#         return 3
#     elif score < 0.1:
#         return 2
#     else:
#         return 1

def get_env_cat_energy(score):
    if score < 0:
        return score
    elif score < 0.00025:
        return 5
    elif score < 0.0005:
        return 4
    elif score < 0.001:
        return 3
    elif score < 0.002:
        return 2
    else:
        return 1

pivot_df.loc[:, ("environment_info_energy", "energy_use_mean")] = pivot_df["environment_info_energy"].mean(axis=1)
pivot_df.loc[:, ("environment_info_energy", "energy_use_category")] = list(map(lambda x: int(get_env_cat_energy(x)), pivot_df["environment_info_energy"]["energy_use_mean"]))

In [ ]:
min_co2 = pivot_df["environment_info_co2"]["co2_emissions_mean"].where(pivot_df["environment_info_co2"]["co2_emissions_mean"] > 0).min()
max_co2 = pivot_df["environment_info_co2"]["co2_emissions_mean"].where(pivot_df["environment_info_co2"]["co2_emissions_mean"] > 0).max()
# pivot_df["environment_info_co2"]["Average CO2"] > 0
max_co2 - min_co2

In [ ]:
min_energy = pivot_df["environment_info_energy"]["energy_use_mean"].where(pivot_df["environment_info_energy"]["energy_use_mean"] > 0).min()
max_energy = pivot_df["environment_info_energy"]["energy_use_mean"].where(pivot_df["environment_info_energy"]["energy_use_mean"] > 0).max()
# pivot_df["environment_info_co2"]["Average CO2"] > 0
print(min_energy, min_energy*1000, max_energy, max_energy*1000, max_energy - min_energy)

In [ ]:
pivot_df["environment_info_co2"][["co2_emissions_mean", "co2_emissions_category"]].join(
pivot_df["environment_info_energy"][["energy_use_mean", "energy_use_category"]])

# Costs

In [ ]:
pivot_df.loc[:, ("costs", "costs_mean")] = pivot_df["costs"][costs_included_benchmarks].mean(axis=1).fillna(-0.001) * 1000

In [ ]:
import math

n_categories = 5
min_costs = pivot_df["costs"]["costs_mean"].where(pivot_df["costs"]["costs_mean"] > 0).min()
# max_costs = pivot_df["costs"]["costs_mean"].where(pivot_df["costs"]["costs_mean"] > 0).max()
# max_costs = min(15, pivot_df["costs"]["costs_mean"].where(pivot_df["costs"]["costs_mean"] > 0).max())
pivot_df.loc[:, ("costs", "costs_zscore")] = zscore(pivot_df["costs"]["costs_mean"], nan_policy="omit")
max_costs = pivot_df["costs"]["costs_mean"].where((pivot_df["costs"]["costs_mean"] > 0) & (pivot_df["costs"]["costs_zscore"] < 3)).max()

intervals = (max_costs - min_costs) / (n_categories - 1)

def get_costs_cat(costs):
    if math.isnan(costs) or not costs:
        return -1
    if costs > max_costs:
        cat = n_categories + 1
    else:
        cat = (min(costs, max_costs) - min_costs) // intervals + 1
    return cat

def get_costs_percent(score):
    percent = min(round(score / max_costs * 100), 100)
    return percent

# pivot_df.loc[:, ("costs", "costs_category")] = pivot_df["costs"]["costs_mean"] * 1000
pivot_df.loc[:, ("costs", "costs_category")] = list(map(lambda x: int(get_costs_cat(x)), pivot_df["costs"]["costs_mean"]))
pivot_df.loc[:, ("costs", "costs_percent")] = list(map(lambda x: int(get_costs_percent(x)), pivot_df["costs"]["costs_mean"]))

In [ ]:
pivot_df["costs"]

# Factuality

In [ ]:
# pivot_df.loc[:, ("score", "Reasoning")] = ((pivot_df["score"]["ARC-NL"] + pivot_df["score"]["MMLU-NL"]) / 2).tolist()
pivot_df.loc[:, ("score", "factuality_mean")] = ((pivot_df["score"]["TinyMMLU"] + pivot_df["score"]["TinyARC"] + pivot_df["score"]["TinyTruthfulQA"]) / 3).tolist()

In [ ]:
def get_factuality_cat(score):
    if score > 0.8:
        return 5
    elif score > 0.7:
        return 4
    elif score > 0.6:
        return 3
    elif score > 0.5:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    
pivot_df.loc[:, ("score", "factuality_category")] = list(map(lambda x: get_factuality_cat(x), pivot_df["score"]["factuality_mean"]))

In [ ]:
# pivot_df["score"][order]
# pivot_df["score"].reindex(columns=order)
# pivot_df["score"][["ARC-NL", "MMLU-NL", "Reasoning", "ReasoningCategory"]]
pivot_df["score"][["TinyMMLU", "TinyARC", "TinyTruthfulQA", "factuality_mean", "factuality_category"]]

# Honesty

In [ ]:
# For now only HonestCityBench
pivot_df.loc[:, ("score", "honesty_mean")] = pivot_df["score"]["HonestCity"]

In [ ]:
def get_honesty_cat(score):
    if score > 0.65:
        return 5
    elif score > 0.5:
        return 4
    elif score > 0.35:
        return 3
    elif score > 0.20:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    
pivot_df.loc[:, ("score", "honesty_category")] = list(map(lambda x: get_honesty_cat(x), pivot_df["score"]["honesty_mean"]))

In [ ]:
pivot_df["score"][["HonestCity", "honesty_mean", "honesty_category"]]

# Simplification

In [ ]:
pivot_df.loc[:, ("score", "simplification_mean")] = ((pivot_df["score"]["AmsterdamSimplification-detailed"] + pivot_df["score"]["INT_Duidelijke_Taal-detailed"]) / 2).tolist()

In [ ]:
def get_simplification_cat_old(score):
    if score > 40:
        return 5
    elif score > 30:
        return 4
    elif score > 20:
        return 3
    elif score > 10:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1

def get_simplification_cat(score):
    if score > 44:
        return 5
    elif score > 38:
        return 4
    elif score > 32:
        return 3
    elif score > 26:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    

pivot_df.loc[:, ("score", "simplification_category")] = list(map(lambda x: get_simplification_cat(x), pivot_df["score"]["simplification_mean"]))

In [ ]:
pivot_df["score"][["AmsterdamSimplification-detailed", "INT_Duidelijke_Taal-detailed", "simplification_mean", "simplification_category"]]

# Summarization

In [ ]:
pivot_df.loc[:, ("score", "summarization_mean")] = ((pivot_df["score"]["CNNDailyMail"] + pivot_df["score"]["XSum"]) / 2).tolist()

In [ ]:
def get_summarization_cat(score):
    if score > 0.65:
        return 5
    elif score > 0.60:
        return 4
    elif score > 0.55:
        return 3
    elif score > 0.50:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    
pivot_df.loc[:, ("score", "summarization_category")] = list(map(lambda x: get_summarization_cat(x), pivot_df["score"]["summarization_mean"]))

In [ ]:
pivot_df["score"][["CNNDailyMail", "XSum", "summarization_mean", "summarization_category"]]

# All scores

In [ ]:
# pd.to_datetime(pivot_df["runtime"]["ARC-NL"]).map(lambda x: x.second)

In [ ]:
final_scores_categories = pivot_df["score"][[
    "factuality_mean", "factuality_category",
    "honesty_mean", "honesty_category",
    "simplification_mean", "simplification_category",
    "summarization_mean", "summarization_category"]].join(
pivot_df["environment_info_energy"][["energy_use_mean", "energy_use_category"]]).join(
pivot_df["costs"][["costs_mean", "costs_category", "costs_percent"]])

In [ ]:
final_scores_categories

In [ ]:
from llm_eval.language_models.llms import llm_config

In [ ]:
name_map = {
    model_config["id"].split("/")[1]: model_name
    for model_name, model_config in llm_config.MODEL_MAPPING.items()
}

name_map.update({
    # "gpt-4o": "GPT-4o",
    # "gpt-4o-mini": "GPT-4o-mini"
    "GPT-4o": "gpt-4o",
    "GPT-4o-mini": "gpt-4o-mini"
})

In [ ]:
# name_map

In [ ]:
import json
existing_data = json.load(open("../llm-eval-website/_data/models.json", "r"))
existing_data[0]

In [ ]:
# aspects = ["factuality", "honesty", "simplification", "summarization", "energy_use", "costs"]
aspects = ["honesty"]

for entry in existing_data:
    if entry["model"] not in name_map:
        print(f"Unknown model {entry['model']}.")
        continue
    model_name_simple = name_map[entry["model"]]
    if model_name_simple not in final_scores_categories.index:
        print(f"Missing new {model_name_simple} scores!!! Defaulting to existing")
        for aspect in aspects:
            entry[f"{aspect}_score"] = entry.pop(f"{aspect}_score", -1)
            entry[f"{aspect}"] = entry.pop(aspect, -1)
        continue
    for aspect in aspects:
        if aspect == "costs":
            entry["costs"] = round(final_scores_categories.loc[model_name_simple, 'costs_mean'],2)
            entry["costs_category"] = round(final_scores_categories.loc[model_name_simple, 'costs_category'],2)
            entry["costs_percent"] = round(final_scores_categories.loc[model_name_simple, 'costs_percent'],2)
        else:
            entry.pop(aspect, -1)
            if f"{aspect}_mean" in final_scores_categories.columns:
                entry[f"{aspect}_score"] = float(final_scores_categories.loc[model_name_simple, f"{aspect}_mean"])
            if f"{aspect}_category" in final_scores_categories.columns:
                entry[f"{aspect}"] = int(final_scores_categories.loc[model_name_simple, f"{aspect}_category"])

In [ ]:
existing_data

In [ ]:
json.dump(existing_data, open("../llm-eval-website/_data/models.json", "w"), default=lambda x: int(x) if isinstance(x, np.integer) else x, indent=4,  ensure_ascii=False)

### Dump results

In [ ]:
%pip install openpyxl

In [ ]:
shared_folder = "/home/azureuser/cloudfiles/code/shared_data/results"
# os.listdir(shared_folder)

In [ ]:
runs = [
    'eurollm_large', 'eurollm_small',
    'falcon_small',
    # 'gemma_large',
    'gemma_large_gossa', 'gemma_small', 'gemma_small_summary', 'gemma_small_tiny',
    'gpt4o', 'gpt4o_mini',
    'llama_small',
    'mistral_tiny', 'mistral_small', 'mistral_small_summary',
    'olmo_large', 'olmo_small', 'olmo_large_summary',
    'phi_mini',
    'qwen_large', 'qwen_small', 'qwen_small_nothinking', 'qwen_large_nothinking',
    'tinyllama',
]

results_files = ["leaderboard_" + run for run in runs]

In [ ]:
results = sum([json.loads(open(f"{shared_folder}/{file}", "rb").read()) for file in results_files], [])

In [ ]:
import pandas as pd
from collections import defaultdict

# This will collect one dataframe per benchmark
benchmark_dfs = defaultdict(dict)
benchmark_df = {}

for result in results:
    benchmark_name = result["metadata"]["benchmark"]["name"]
    model_name = result["metadata"]["llm"]["model_name"]
    
    outputs = result["benchmark_results"]["run_output"]
    for idx, sample in enumerate(outputs):
        source = sample["source"] if "source" in sample else sample["input"]
        target = sample["target"] if "target" in sample else sample["summary"]
        response = sample["response"]
        
        if source not in benchmark_dfs[benchmark_name]:
            benchmark_dfs[benchmark_name][source] = {}
            benchmark_dfs[benchmark_name][source]["source"] = source
            benchmark_dfs[benchmark_name][source]["target"] = target
        benchmark_dfs[benchmark_name][source][model_name] = response

        if source not in benchmark_df:
            benchmark_df[source] = {}
            benchmark_df[source]["target"] = target
            benchmark_df[source]["benchmark"] = benchmark_name
        benchmark_df[source][model_name] = response

# Convert to pandas DataFrames
dfs = {}
for benchmark_name, data in benchmark_dfs.items():
    df = pd.DataFrame.from_dict(data, orient='index')
    df.index.name = "source"
    dfs[benchmark_name] = df

    df.to_csv(f"{benchmark_name}.csv")

order = ["TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", 
         "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]
df = pd.DataFrame.from_dict(benchmark_df, orient='index')
df.index.name = "source"
df['benchmark'] = pd.Categorical(df['benchmark'], categories=order, ordered=True)
df = df.sort_values('benchmark')
df.to_csv(f"all_benchmarks.csv")
df.to_excel(f"all_benchmarks.xlsx")